In [ ]:
using DataDrivenDiffEq
using ModelingToolkit
using OrdinaryDiffEq
using DataDrivenSparse
using LinearAlgebra
using NCDatasets
using CairoMakie

ROOT = normpath(joinpath(@__DIR__, "..", "..")) 
data_path = joinpath(ROOT, "pacco", "output", "BASE", "BASE.nc")
output_path = joinpath(ROOT, "analysis", "pacco_validation", "output")

# Load ice volume
ds = NCDataset(data_path)
exclude = ["time", "lat", "lon"]
target_vars = filter(v -> !(v in exclude), collect(keys(ds)))

# Build Matrix (Rows = Variables, Cols = Time)
X = reduce(vcat, [reshape(Array(ds[v][:]), 1, :) for v in target_vars])
X = Float64.(replace(X, missing => 0.0))
t = Array(ds["time"][:])
close(ds)

# --- Automated Symbolic Setup ---
@parameters t
@variables x(t)[1:length(target_vars)]
u_vars = collect(x)
basis = Basis(polynomial_basis(u_vars, 2), u_vars, iv = t)
opt = STLSQ(0.01)

# 2. Create the problem using matrices instead of the 'sol' object
# This skips the 'is_successful' check entirely
ddprob = ContinuousDataDrivenProblem(X, t)


# Try solving without the 'options' argument first to keep it simple
ddsol = solve(ddprob, basis, opt)


In [27]:
print(ddsol.basis)
print(get_parameter_values(ddsol.basis))

Model ##Basis#306 with 0 equations
States : x(t)
Independent variable: t
Equations
Float64[]

In [ ]:
using DataDrivenDiffEq
using ModelingToolkit
using OrdinaryDiffEq
using DataDrivenSparse
using LinearAlgebra
using NCDatasets
using CairoMakie

ROOT = normpath(joinpath(@__DIR__, "..", ".."))
data_path = joinpath(ROOT, "pacco", "output", "BASE", "BASE.nc")
output_path = joinpath(ROOT, "analysis", "pacco_validation", "output")
mkpath(output_path)

# ─── 1. Load Data ────────────────────────────────────────────────────────────
println("📂 Loading NetCDF data...")
ds = NCDataset(data_path)

exclude = ["time", "lat", "lon"]
target_vars = filter(v -> !(v in exclude), collect(keys(ds)))
println("  → Variables found ($(length(target_vars))): ", join(target_vars, ", "))

t_raw = Array(ds["time"][:])
X_raw = reduce(vcat, [reshape(Array(ds[v][:]), 1, :) for v in target_vars])
X_raw = Float64.(replace(X_raw, missing => 0.0))
close(ds)

println("  → Data matrix shape: $(size(X_raw)) (vars × time)")
println("  → Time span: $(t_raw[1]) to $(t_raw[end]), N=$(length(t_raw))")

# ─── 2. DEBUG: Inspect data quality ──────────────────────────────────────────
println("\n🔍 Data diagnostics:")
for (i, v) in enumerate(target_vars)
    row = X_raw[i, :]
    n_zero = sum(row .== 0.0)
    println("  [$i] $v: min=$(round(minimum(row), sigdigits=4)), max=$(round(maximum(row), sigdigits=4)), zeros=$n_zero/$(length(row))")
end

# ─── 3. Filter out near-constant or all-zero variables ───────────────────────
println("\n🧹 Filtering variables...")
variances = [var(X_raw[i, :]) for i in 1:size(X_raw, 1)]
active_mask = variances .> 1e-10
active_vars = target_vars[active_mask]
X_active = X_raw[active_mask, :]

println("  → Kept $(sum(active_mask)) / $(length(target_vars)) variables with sufficient variance")
println("  → Dropped: ", join(target_vars[.!active_mask], ", "))

# ─── 4. Normalize data to [-1, 1] per variable ───────────────────────────────
println("\n📏 Normalizing data...")
X_norm = similar(X_active)
scales = zeros(size(X_active, 1))
offsets = zeros(size(X_active, 1))

for i in 1:size(X_active, 1)
    lo, hi = minimum(X_active[i, :]), maximum(X_active[i, :])
    rng = hi - lo
    offsets[i] = (hi + lo) / 2
    scales[i] = rng < 1e-12 ? 1.0 : rng / 2
    X_norm[i, :] = (X_active[i, :] .- offsets[i]) ./ scales[i]
end


# ─── 6. Build symbolic basis ──────────────────────────────────────────────────
# CRITICAL: polynomial_basis(u_vars, 2) with N variables → O(N²) terms
# For N > ~10 this is very slow. Use degree=1 (linear) or reduce N first.
println("\n🧮 Building symbolic basis ($(length(active_vars)) variables)...")

n_vars = length(active_vars)
@parameters τ                          # use τ to avoid shadowing Base.t
@variables u(τ)[1:n_vars]
u_vars = collect(u)

poly_degree = n_vars > 8 ? 1 : 2      # auto-reduce degree if many vars
println("  → Polynomial degree: $poly_degree ($(n_vars) active variables)")
println("  → Basis library size ≈ $(binomial(n_vars + poly_degree, poly_degree)) terms")

basis = Basis(polynomial_basis(u_vars, poly_degree), u_vars, iv = τ)
println("  → Basis built: $(length(basis)) terms")

# ─── 7. Solve SINDy ──────────────────────────────────────────────────────────
println("\n🚀 Setting up DataDrivenProblem...")
ddprob = ContinuousDataDrivenProblem(X_ds, t_ds)

# STLSQ threshold: too small → dense model; too large → zero model
# Start at 0.1 on normalized data, then tune
thresholds = [0.001, 0.01, 0.05, 0.1, 0.2]

println("  → Trying STLSQ thresholds: $thresholds")
best_sol = nothing
best_thresh = nothing
best_residual = Inf

for thresh in thresholds
    println("     Testing threshold = $thresh ...")
    opt_local = STLSQ(thresh)
    try
        sol = solve(ddprob, basis, opt_local)
        # ModelingToolkit result check
        res = result(sol)
        nnz = count(!iszero, res)
        r2  = rss(sol)  # residual sum of squares
        println("       non-zeros=$nnz, RSS=$( round(r2, sigdigits=4))")
        if r2 < best_residual && nnz > 0
            best_residual = r2
            best_sol = sol
            best_thresh = thresh
        end
    catch e
        println("       ⚠️  Failed: $e")
    end
end

# ─── 8. Report results ────────────────────────────────────────────────────────
if isnothing(best_sol)
    println("\n❌ All thresholds failed. Try: lower poly_degree, more downsampling, or larger threshold.")
else
    println("\n✅ Best result at threshold=$best_thresh, RSS=$(round(best_residual, sigdigits=4))")
    println("\n📋 Discovered equations:")
    println(result(best_sol))

    # Save discovered equations to file
    eq_path = joinpath(output_path, "discovered_equations.txt")
    open(eq_path, "w") do f
        println(f, "SINDy results — threshold=$best_thresh")
        println(f, "Variables: ", join(active_vars, ", "))
        println(f, "\n", result(best_sol))
    end
    println("  → Saved to $eq_path")
end

📂 Loading NetCDF data...
  → Variables found (43): T, C, iceage, albedo, H, Hsed, B, Tice, fstr, I, Tsl, Tref, Cref, z, Surf, Vol, albedo_ref, Tsurf, s, a, taud, taub, vd, vb, fstr_ref, beta, v, hcond, hvadv, hdrag, hgeo, L, Hb, Pe, albedo_eff, precipitation, Inorm, Ianom, m, q, sw, lw, RCO2
  → Data matrix shape: (43, 3001) (vars × time)
  → Time span: -3.0e6 to 0.0, N=3001

🔍 Data diagnostics:
  [1] T: min=257.6, max=279.1, zeros=0/3001
  [2] C: min=201.3, max=309.7, zeros=0/3001
  [3] iceage: min=-3.138, max=106500.0, zeros=1/3001
  [4] albedo: min=0.2, max=0.9, zeros=0/3001
  [5] H: min=-1.658, max=2036.0, zeros=1/3001
  [6] Hsed: min=5.005, max=30.13, zeros=0/3001
  [7] B: min=-176.4, max=502.1, zeros=0/3001
  [8] Tice: min=258.2, max=273.2, zeros=0/3001
  [9] fstr: min=0.2, max=0.4, zeros=0/3001
  [10] I: min=429.6, max=564.9, zeros=0/3001
  [11] Tsl: min=0.0, max=0.0, zeros=3001/3001
  [12] Tref: min=273.1, max=273.2, zeros=0/3001
  [13] Cref: min=280.0, max=280.0, zeros=0/3001


UndefVarError: UndefVarError: `var` not defined in `Main`
Suggestion: check for spelling errors or missing imports.
Hint: a global variable of this name also exists in Statistics.
    - Also exported by StatsBase (loaded but not imported in Main).
    - Also exported by Distributions (loaded but not imported in Main).